# 00 — Gym basics without magic

Goal: make `reset()` and `step()` boring.

We will build **LineWorld**:

```text
0 -- 1 -- 2 -- 3 -- 4 -- 5
S                        G
```

The agent starts at 0. It can go LEFT or RIGHT.

- reaching 5 gives `+1`
- each step costs `-0.02`
- falling off the left edge keeps you at 0
- the episode ends at 5 or after 30 steps

This is a tiny MDP where we can see every variable.

In [ ]:
!pip -q install "gymnasium>=1.0" numpy pandas matplotlib

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

class LineWorld(gym.Env):
    def __init__(self):
        super().__init__()
        self.action_space = spaces.Discrete(2)   # 0=LEFT, 1=RIGHT
        self.observation_space = spaces.Discrete(6)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.position = 0          # <-- true environment state
        self.steps = 0
        obs = self.position        # <-- observation returned to agent
        return obs, {}

    def step(self, action):
        self.steps += 1

        # TRANSITION: how the world changes
        if action == 0:
            self.position = max(0, self.position - 1)
        else:
            self.position = min(5, self.position + 1)

        # REWARD
        terminated = self.position == 5
        reward = 1.0 if terminated else -0.02
        truncated = self.steps >= 30

        obs = self.position
        return obs, reward, terminated, truncated, {}

env = LineWorld()
obs, info = env.reset()
print("After reset(), first observation =", obs)


## One manual `step`

`reset()` did **not** mean “there is no state.”

It initialized the hidden/internal world and returned the first observation.

Now take RIGHT (`action=1`).

In [ ]:
old_obs = obs
action = 1
obs, reward, terminated, truncated, info = env.step(action)

print("old observation:", old_obs)
print("action:", "RIGHT")
print("new observation:", obs)
print("reward:", reward)
print("terminated:", terminated)


## You can be the policy

A **policy** is simply the thing choosing actions.

Here, *you* can replace the policy mentally:

```text
observation -> YOU -> LEFT/RIGHT
```

A stupid policy can also be code.

In [ ]:
def random_policy(obs):
    return env.action_space.sample()

obs, _ = env.reset()
trajectory = []

for t in range(30):
    action = random_policy(obs)
    next_obs, reward, terminated, truncated, _ = env.step(action)
    trajectory.append((t, obs, action, reward, next_obs))
    obs = next_obs
    if terminated or truncated:
        break

pd.DataFrame(trajectory, columns=["t","obs","action","reward","next_obs"])


## A hand-written policy

This policy is not trained at all:

```python
always move RIGHT
```

It is still a policy because it maps observation/state → action.

In [ ]:
def always_right(obs):
    return 1

obs, _ = env.reset()
total_reward = 0
trajectory = []

for t in range(30):
    action = always_right(obs)
    next_obs, reward, terminated, truncated, _ = env.step(action)
    trajectory.append((t, obs, "RIGHT", reward, next_obs))
    total_reward += reward
    obs = next_obs
    if terminated or truncated:
        break

print("Return:", total_reward)
pd.DataFrame(trajectory, columns=["t","obs","action","reward","next_obs"])


## Now actually learn a policy with Q-learning

Instead of writing “go right,” we store a value for each `(state, action)`.

At first the values are all zero.

Then repeated experience changes them.

In [ ]:
Q = np.zeros((6, 2))
alpha = 0.2
gamma = 0.95
epsilon = 0.25

episode_returns = []

for episode in range(300):
    obs, _ = env.reset()
    total = 0

    while True:
        # POLICY: epsilon-greedy from current Q values
        if np.random.rand() < epsilon:
            action = env.action_space.sample()
        else:
            action = int(np.argmax(Q[obs]))

        next_obs, reward, terminated, truncated, _ = env.step(action)
        total += reward

        # LEARNING: update the value of the action we just took
        future = 0 if terminated else np.max(Q[next_obs])
        target = reward + gamma * future
        Q[obs, action] += alpha * (target - Q[obs, action])

        obs = next_obs
        if terminated or truncated:
            break

    episode_returns.append(total)

print("Learned Q table:")
print(pd.DataFrame(Q, columns=["LEFT", "RIGHT"]))


In [ ]:
plt.figure(figsize=(8,4))
plt.plot(pd.Series(episode_returns).rolling(20, min_periods=1).mean())
plt.xlabel("Episode")
plt.ylabel("20-episode rolling return")
plt.title("LineWorld learning")
plt.show()

print("Learned action at every position:")
for s in range(5):
    print(s, "->", "RIGHT" if np.argmax(Q[s]) == 1 else "LEFT")


### What just happened?

You now have the entire RL loop:

```text
S_t / observation
      ↓
policy chooses A_t
      ↓
environment transition
      ↓
S_(t+1)
      ↓
reward R_(t+1)
      ↓
Q values change
      ↓
future policy changes
```

Pong, Atari, Dota, τ², and Sea of Stars add complexity to these boxes. They do not replace this loop.
